In [1]:
from pathlib import Path 
import cv2
import os

from scipy import ndimage
import numpy as np 
from PIL import Image
import matplotlib.pyplot as plt

from encoder import ThermalEncoder
import torch 
import torch.nn as nn
from pathlib import Path
import torchvision.datasets as datasets

/Users/fred/Documents/University of Leeds/Year 3/1/Dist/repo/Thermal_edge_detection/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:

data_path = Path("/Volumes/Samsung_1TB/thermal_images/archive/").expanduser()

pairs = []

for visible_path in data_path.glob("set*/V*/visible/*.jpg"):
    thermal_path = visible_path.parents[1] / "lwir" / visible_path.name
    
    if thermal_path.exists():
        pairs.append((visible_path, thermal_path))

print(f"Total pairs found: {len(pairs)}")

Total pairs found: 47264


In [10]:
def weighted_cross_entropy_loss(prediction, label):
    """
    HED-specific loss to handle edge/non-edge imbalance.
    """
    label = label.float()
    mask = (label > 0.5).float()
    
    num_positive = torch.sum(mask)
    num_negative = torch.sum(1 - mask)
    
    # beta is the ratio of non-edge pixels
    beta = num_negative / (num_positive + num_negative)
    
    # Use binary_cross_entropy_with_logits for numerical stability
    cost = F.binary_cross_entropy_with_logits(prediction, label, reduction='none')
    
    # Weight the edge pixels higher (beta) and non-edge lower (1-beta)
    weighted_cost = beta * cost * mask + (1 - beta) * cost * (1 - mask)
    return torch.mean(weighted_cost)

In [11]:
def train_bridge(pairs, hed_net, epochs=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    bridge = ThermalToHEDBridge().to(device)
    optimizer = optim.Adam(bridge.parameters(), lr=1e-3)
    
    dataset = ThermalVisibleDataset(pairs)
    loader = DataLoader(dataset, batch_size=4, shuffle=True)
    
    # Freeze HED (teacher model)
    hed_net.eval()
    for param in hed_net.parameters():
        param.requires_grad = False

    for epoch in range(epochs):
        total_loss = 0
        
        for therm, vis in loader:
            therm, vis = therm.to(device), vis.to(device)
            
            # Teacher: edges from visible image
            with torch.no_grad():
                vis_3ch = vis.repeat(1, 3, 1, 1)
                edge_gt = hed_net(vis_3ch)

            # Student: thermal → bridge → HED
            pseudo_rgb = bridge(therm)
            edge_pred = hed_net(pseudo_rgb)
            
            # Loss
            loss = weighted_cross_entropy_loss(edge_pred, edge_gt)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(loader)}")
    
    return bridge

In [12]:
class ThermalVisibleDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        vis_path, therm_path = self.pairs[idx]
        
        # Load and normalize (0-1)
        vis = cv2.imread(str(vis_path), cv2.IMREAD_GRAYSCALE)
        therm = cv2.imread(str(therm_path), cv2.IMREAD_GRAYSCALE)
        
        vis = cv2.normalize(vis, None, 0, 1, cv2.NORM_MINMAX, dtype=cv2.CV_32F)
        therm = cv2.normalize(therm, None, 0, 1, cv2.NORM_MINMAX, dtype=cv2.CV_32F)
        
        return torch.tensor(therm).unsqueeze(0), torch.tensor(vis).unsqueeze(0)

NameError: name 'Dataset' is not defined

In [ ]:
Dataset = 